# NHANES Meal Archetypes Walkthrough

This notebook reproduces the first-pass NHANES meal-archetype workflow used to create `analysis/nhanes/nhanes_meal_archetypes.md`.

It is organized so you can:
- inspect the helper code,
- run the data preparation and meal summarization steps,
- compare top archetypes across SES groups, and
- regenerate the markdown note.


## 1. Helper Code

This cell contains the imports, file locations, category mappings, and helper functions used by the analysis.


In [1]:
# pyright: reportGeneralTypeIssues=false, reportAttributeAccessIssue=false, reportAssignmentType=false, reportArgumentType=false

from __future__ import annotations

import io
import re
from typing import cast
from pathlib import Path

import numpy as np
import pandas as pd
import requests


ROOT = Path.cwd()
if not (ROOT / "scripts" / "nhanes_meal_archetypes.py").exists():
    if (ROOT.parent.parent / "scripts" / "nhanes_meal_archetypes.py").exists():
        ROOT = ROOT.parent.parent
    else:
        raise FileNotFoundError("Run this notebook from the repo root or from analysis/nhanes.")
OUT_DIR = ROOT / "analysis" / "nhanes"
RAW_DIR = OUT_DIR / "raw"

URLS = {
    "demo": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.XPT",
    "dr1iff": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DR1IFF_L.XPT",
    "drxfd": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DRXFCD_L.XPT",
    "foodcat": "https://www.ars.usda.gov/ARSUserFiles/80400530/apps/WWEIA_August2021_August2023_foodcat_FNDDS.xlsx",
}


OCCASION_MAP = {
    1: "breakfast",
    5: "breakfast",
    10: "breakfast",
    2: "lunch",
    11: "lunch",
    3: "dinner",
    4: "dinner",
    12: "dinner",
    14: "dinner",
    6: "snack",
    13: "snack",
    15: "snack",
    16: "snack",
    17: "snack",
    18: "snack",
    7: "beverage",
    19: "beverage",
    8: "other",
    9: "other",
    91: "other",
}


COMBO_MAP = {
    1: "drink_additions",
    2: "cereal_combo",
    3: "bread_combo",
    4: "salad_combo",
    5: "sandwich_combo",
    6: "soup_combo",
    7: "frozen_meal_combo",
    8: "ice_cream_combo",
    9: "beans_veg_combo",
    10: "fruit_combo",
    11: "tortilla_combo",
    12: "protein_combo",
    13: "lunchables_combo",
    14: "chips_combo",
    15: "baby_combo",
    90: "mixed_combo",
}


GROUP_PATTERNS = [
    ("coffee_tea", r"coffee|tea"),
    ("water", r"\bwater\b"),
    ("alcohol", r"beer|wine|liquor|alcohol|cocktail"),
    ("sweet_drink", r"soft drinks|sports.*drink|energy drink|sweetened beverage|fruit drink|fruit ades|sports beverage"),
    ("juice_smoothie", r"juice|smoothie"),
    ("pizza", r"pizza"),
    ("tortilla", r"taco|burrito|quesadilla|enchilada|tortilla|mexican mixed dishes"),
    ("sandwich", r"sandwich|burger|hot dog|slider"),
    ("salad", r"salad"),
    ("soup", r"soup"),
    ("eggs", r"egg|omelet|omelette"),
    ("cereal_oatmeal", r"cereal|oatmeal|grits"),
    ("yogurt", r"yogurt"),
    ("milk_dairy_drink", r"milk|milk shakes|dairy drinks|plant-based milk"),
    ("cheese", r"cheese"),
    ("sweet_baked", r"sweet bakery|cakes|cookies|pies|doughnuts|pastries|croissants|muffins|ice cream|frozen dairy desserts|candy|chocolate|dessert|brownies|sweet rolls|snack bars"),
    ("bread", r"bread|rolls|bagels|english muffins|biscuits|toast|quick breads|pancakes|waffles|french toast"),
    ("breakfast_meat", r"bacon|sausage|ham"),
    ("seafood", r"fish|shellfish|seafood"),
    ("meat_poultry", r"beef|pork|chicken|turkey|lamb|goat|meat|organ meats|deli meat|frankfurter|meat mixed dishes"),
    ("beans_nuts", r"beans|legumes|lentils|nuts|seeds|peanut butter|nut butter"),
    ("grain_starch", r"pasta|rice|noodles|macaroni|grains|grain dishes|potatoes|fried potatoes|starchy vegetables|corn"),
    ("fruit", r"fruit|apples|bananas|berries|citrus|grapes|melon|peaches|pears|pineapple|raisins|applesauce"),
    ("vegetables", r"vegetables|vegetable mixed dishes|greens|broccoli|carrots|tomatoes|lettuce|beans, peas|peas and carrots|other vegetables"),
    ("salty_snack", r"chips|crackers|pretzels|popcorn|snack"),
    ("mixed_main", r"mixed dishes|casseroles|stir fry|fried rice|lo mein|macaroni and cheese|frozen meals|asian mixed dishes|italian mixed dishes"),
]


def ensure_dirs() -> None:
    RAW_DIR.mkdir(parents=True, exist_ok=True)


def download(url: str, path: Path) -> bytes:
    if path.exists():
        return path.read_bytes()
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
    response.raise_for_status()
    path.write_bytes(response.content)
    return response.content


def load_xpt(name: str) -> pd.DataFrame:
    content = download(URLS[name], RAW_DIR / f"{name}.xpt")
    return cast(pd.DataFrame, pd.read_sas(io.BytesIO(content), format="xport", encoding="utf-8"))


def load_food_categories() -> pd.DataFrame:
    content = download(URLS["foodcat"], RAW_DIR / "food_categories.xlsx")
    return pd.read_excel(io.BytesIO(content), sheet_name="Aug2021-Aug2023_FNDDS_foodcat")


def clean_numeric(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.mask(numeric.abs() < 1e-20, 0)


def normalize_occasion(value: float) -> str:
    if pd.isna(value):
        return "other"
    return OCCASION_MAP.get(int(value), "other")


def food_group(category_description: str) -> str:
    text = str(category_description).lower()
    for group, pattern in GROUP_PATTERNS:
        if re.search(pattern, text):
            return group
    return "other"


def top_items(values: pd.Series, limit: int = 4) -> str:
    counts = values.value_counts()
    return ", ".join(map(str, counts.index[:limit].tolist()))


def compact_list(text: str, limit: int = 4) -> str:
    if not text:
        return "-"
    seen: list[str] = []
    for part in [item.strip() for item in text.split(",") if item.strip()]:
        if part not in seen:
            seen.append(part)
        if len(seen) >= limit:
            break
    return ", ".join(seen) if seen else "-"


def stratum_label(name: str) -> str:
    labels = {
        "all": "All Respondents",
        "pir_3plus": "Higher SES (PIR >= 3)",
        "pir_5plus": "Highest SES (PIR >= 5)",
        "pir_lt_1_3": "Lower SES (PIR < 1.3)",
    }
    return labels.get(name, name)


def build_label(occasion: str, groups: set[str], combos: set[str]) -> str:
    if occasion == "breakfast":
        if "cereal_combo" in combos or "cereal_oatmeal" in groups:
            return "cereal or oatmeal breakfast"
        if "sandwich_combo" in combos or "sandwich" in groups or ("tortilla_combo" in combos and ("eggs" in groups or "meat_poultry" in groups)):
            return "breakfast sandwich or burrito"
        if "eggs" in groups and ("bread" in groups or "breakfast_meat" in groups):
            return "eggs with bread or meat"
        if "coffee_tea" in groups and ("sweet_baked" in groups or "bread" in groups):
            return "coffee and pastry breakfast"
        if "yogurt" in groups and "fruit" in groups:
            return "yogurt and fruit breakfast"
        if "fruit" in groups and "bread" in groups:
            return "fruit and bread breakfast"
        if "coffee_tea" in groups:
            return "coffee-first breakfast"
        return "other breakfast"

    if occasion == "lunch":
        if "sandwich_combo" in combos or "sandwich" in groups:
            return "sandwich lunch"
        if "salad_combo" in combos or "salad" in groups:
            return "salad lunch"
        if "soup_combo" in combos or "soup" in groups:
            return "soup lunch"
        if "tortilla_combo" in combos or "tortilla" in groups:
            return "tortilla lunch"
        if "pizza" in groups:
            return "pizza lunch"
        if {"meat_poultry", "seafood"} & groups and {"grain_starch", "vegetables"} & groups:
            return "plate lunch"
        if "mixed_main" in groups or "mixed_combo" in combos:
            return "mixed entree lunch"
        return "other lunch"

    if occasion == "dinner":
        if "tortilla_combo" in combos or "tortilla" in groups:
            return "tortilla dinner"
        if "pizza" in groups:
            return "pizza dinner"
        if "soup_combo" in combos or "soup" in groups:
            return "soup dinner"
        if {"meat_poultry", "seafood", "beans_nuts"} & groups and {"grain_starch", "vegetables"} <= groups:
            return "protein vegetable starch dinner"
        if {"meat_poultry", "seafood", "beans_nuts"} & groups and "grain_starch" in groups:
            return "protein and starch dinner"
        if "mixed_main" in groups or "mixed_combo" in combos or "frozen_meal_combo" in combos:
            return "mixed entree dinner"
        if "sandwich_combo" in combos or "sandwich" in groups:
            return "sandwich dinner"
        return "other dinner"

    if occasion == "snack":
        if "chips_combo" in combos or "salty_snack" in groups:
            return "salty snack"
        if "sweet_baked" in groups:
            return "sweet snack"
        if "fruit" in groups and ("yogurt" in groups or "cheese" in groups):
            return "fruit and dairy snack"
        if "sandwich_combo" in combos or "sandwich" in groups:
            return "sandwich snack"
        if "coffee_tea" in groups:
            return "coffee or tea snack"
        return "other snack"

    return "other occasion"


def build_strata(participants: pd.DataFrame) -> dict[str, pd.Series]:
    pir = participants["INDFMPIR"]
    return {
        "all": pd.Series(True, index=participants.index),
        "pir_3plus": pir >= 3,
        "pir_5plus": pir >= 5,
        "pir_lt_1_3": pir < 1.3,
    }


## 2. Core Transformation Functions

This cell defines the main data-preparation and summarization functions.


In [2]:
def prepare_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    demo = load_xpt("demo")
    dr1 = load_xpt("dr1iff")
    drxfd = load_xpt("drxfd")
    foodcats = load_food_categories()

    demo["SEQN"] = clean_numeric(demo["SEQN"]).astype("Int64")
    demo["INDFMPIR"] = clean_numeric(demo["INDFMPIR"])
    demo["RIDAGEYR"] = clean_numeric(demo["RIDAGEYR"])

    use_cols = [
        "SEQN",
        "WTDRD1",
        "DR1_020",
        "DR1_030Z",
        "DR1CCMTX",
        "DR1IFDCD",
        "DR1IKCAL",
        "DR1FS",
        "DR1_040Z",
    ]
    dr1 = dr1[use_cols].copy()
    for col in use_cols:
        dr1[col] = clean_numeric(dr1[col])

    dr1["SEQN"] = dr1["SEQN"].astype("Int64")
    dr1["food_code"] = dr1["DR1IFDCD"].round().astype("Int64")
    dr1["occasion"] = dr1["DR1_030Z"].apply(normalize_occasion)
    dr1 = dr1[dr1["occasion"].isin(["breakfast", "lunch", "dinner", "snack"])]

    foodcats["food_code"] = clean_numeric(foodcats["food_code"]).round().astype("Int64")
    drxfd["food_code"] = clean_numeric(drxfd["DRXFDCD"]).round().astype("Int64")
    drxfd["food_description"] = drxfd["DRXFCLD"]
    drxfd = drxfd[["food_code", "food_description"]]
    foodcats = foodcats[["food_code", "category_number", "category_description"]]

    dr1 = dr1.merge(foodcats, on="food_code", how="left")
    dr1 = dr1.merge(drxfd, on="food_code", how="left")
    dr1["category_description"] = dr1["category_description"].fillna(dr1["food_description"])
    dr1["broad_group"] = dr1["category_description"].apply(food_group)
    dr1["combo_name"] = dr1["DR1CCMTX"].round().astype(int).map(COMBO_MAP).fillna("none")

    participants = demo.loc[demo["SEQN"].notna(), ["SEQN", "INDFMPIR", "RIDAGEYR"]].copy()
    dr1 = dr1.merge(participants, on="SEQN", how="left")
    dr1["meal_key"] = (
        dr1["SEQN"].astype(str)
        + "|"
        + dr1["DR1_020"].round().astype(int).astype(str)
        + "|"
        + dr1["occasion"]
    )

    meal_base = (
        dr1.groupby("meal_key", as_index=False)
        .agg(
            SEQN=("SEQN", "first"),
            occasion=("occasion", "first"),
            meal_time=("DR1_020", "first"),
            weight=("WTDRD1", "first"),
            total_kcal=("DR1IKCAL", "sum"),
            pir=("INDFMPIR", "first"),
            age=("RIDAGEYR", "first"),
        )
    )

    meal_groups = (
        dr1.groupby(["meal_key", "broad_group"], as_index=False)
        .agg(group_kcal=("DR1IKCAL", "sum"), group_items=("food_code", "count"))
    )
    meal_groups = meal_groups[meal_groups["broad_group"] != "other"]
    group_summary = (
        meal_groups.sort_values(by="group_items")  # type: ignore[call-overload]
        .sort_values(by="group_kcal", ascending=False)
        .sort_values(by="meal_key")
        .groupby("meal_key")
        .agg(
            top_groups=("broad_group", lambda s: ", ".join(s.head(4))),
            group_set=("broad_group", lambda s: sorted(set(s))),
        )
        .reset_index()
    )

    combo_summary = (
        dr1.loc[dr1["combo_name"] != "none"]
        .groupby("meal_key")
        .agg(combo_set=("combo_name", lambda s: sorted(set(s))))
        .reset_index()
    )

    category_summary = (
        dr1.groupby(["meal_key", "category_description"], as_index=False)
        .agg(cat_kcal=("DR1IKCAL", "sum"))
        .sort_values(by="cat_kcal", ascending=False)  # type: ignore[call-overload]
        .sort_values(by="meal_key")
        .groupby("meal_key")
        .agg(top_categories=("category_description", lambda s: ", ".join(s.head(5))))
        .reset_index()
    )

    meals = meal_base.merge(group_summary, on="meal_key", how="left")
    meals = meals.merge(combo_summary, on="meal_key", how="left")
    meals = meals.merge(category_summary, on="meal_key", how="left")
    meals["group_set"] = meals["group_set"].apply(lambda x: x if isinstance(x, list) else [])
    meals["combo_set"] = meals["combo_set"].apply(lambda x: x if isinstance(x, list) else [])
    meals["top_groups"] = meals["top_groups"].fillna("")
    meals["top_categories"] = meals["top_categories"].fillna("")
    meals["archetype"] = meals.apply(
        lambda row: build_label(row["occasion"], set(row["group_set"]), set(row["combo_set"])),
        axis=1,
    )
    return dr1, meals


def summarize(meals: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    participant_frame = meals[["SEQN", "pir", "age"]].drop_duplicates().reset_index(drop=True)
    participant_strata_frame = participant_frame.copy()
    participant_strata_frame["INDFMPIR"] = participant_strata_frame["pir"]
    strata = build_strata(participant_strata_frame)

    participant_rows = []
    summary_frames = []
    meal_strata_frame = meals.copy()
    meal_strata_frame["INDFMPIR"] = meal_strata_frame["pir"]
    meal_strata = build_strata(meal_strata_frame)

    for name, mask in strata.items():
        subset = participant_frame.loc[mask.values]
        participant_rows.append(
            {
                "stratum": name,
                "participants": subset["SEQN"].nunique(),
                "mean_age": round(subset["age"].mean(), 1) if len(subset) else np.nan,
            }
        )

    for name, mask in meal_strata.items():
        subset = meals.loc[mask.values].copy()
        if subset.empty:
            continue
        occasion_totals = subset.groupby("occasion")["weight"].sum().rename("occasion_weight")
        archetype = (
            subset.groupby(["occasion", "archetype"], as_index=False)
            .agg(
                weighted_meals=("weight", "sum"),
                unweighted_meals=("meal_key", "nunique"),
                participants=("SEQN", "nunique"),
                top_groups=("top_groups", top_items),
                top_categories=("top_categories", top_items),
            )
            .merge(occasion_totals, on="occasion", how="left")
        )
        archetype["weighted_share"] = archetype["weighted_meals"] / archetype["occasion_weight"]
        archetype["stratum"] = name
        summary_frames.append(archetype)

    summary = pd.concat(summary_frames, ignore_index=True)
    summary = summary.sort_values(by=["stratum", "occasion", "weighted_meals"], ascending=[True, True, False])
    participants = pd.DataFrame(participant_rows)
    return participants, summary


## 3. Markdown Export Function

This is the formatter used to build the Obsidian-friendly markdown note.


In [3]:
def write_markdown(participants: pd.DataFrame, summary: pd.DataFrame) -> None:
    lines = []
    lines.append("# NHANES Meal Archetypes (August 2021-August 2023)")
    lines.append("")
    lines.append("This note summarizes a first-pass meal-archetype extraction using NHANES/WWEIA Day 1 dietary recall data and NHANES demographics.")
    lines.append("")
    lines.append("## What This Is")
    lines.append("")
    lines.append("- Meal unit: one respondent, one day, one eating occasion")
    lines.append("- Source files: NHANES demographics plus WWEIA individual foods")
    lines.append("- Weighting: Day 1 dietary weight (`WTDRD1`)")
    lines.append("- SES split: poverty-income ratio (`INDFMPIR`)")
    lines.append("- Caveat: these are heuristic meal archetypes, so the `other` buckets are still broad")
    lines.append("")
    lines.append("## Sample Sizes")
    lines.append("")
    lines.append("| Group | Participants | Mean age |")
    lines.append("|---|---:|---:|")
    for row in participants.itertuples(index=False):
        mean_age = "" if pd.isna(row.mean_age) else f"{row.mean_age:.1f}"
        lines.append(f"| {stratum_label(row.stratum)} | {row.participants:,} | {mean_age} |")

    for stratum in ["all", "pir_3plus", "pir_5plus", "pir_lt_1_3"]:
        sub = summary[summary["stratum"] == stratum]
        if sub.empty:
            continue
        people = participants[participants["stratum"] == stratum].iloc[0]
        lines.append("")
        lines.append(f"## {stratum_label(stratum)}")
        lines.append("")
        lines.append(f"- Participants: {int(people['participants']):,}")
        if not pd.isna(people["mean_age"]):
            lines.append(f"- Mean age: {people['mean_age']:.1f}")
        for occasion in ["breakfast", "lunch", "dinner", "snack"]:
            occ = sub[sub["occasion"] == occasion].head(5)
            if occ.empty:
                continue
            lines.append("")
            lines.append(f"### {occasion.capitalize()}")
            lines.append("")
            for idx, row in enumerate(occ.itertuples(index=False), start=1):
                groups = compact_list(row.top_groups, limit=4)
                categories = compact_list(row.top_categories, limit=4)
                lines.append(
                    f"{idx}. **{row.archetype}** - {row.weighted_share:.1%} weighted share; {row.participants:,} participants"
                )
                lines.append(f"   - Typical groups: {groups}")
                lines.append(f"   - Typical foods: {categories}")
            other_share = sub[(sub["occasion"] == occasion) & (~sub["archetype"].isin(occ["archetype"]))]["weighted_share"].sum()
            if other_share > 0:
                lines.append(f"   - Remaining archetypes: {other_share:.1%} of {occasion} weight")

    (OUT_DIR / "nhanes_meal_archetypes.md").write_text("\n".join(lines), encoding="utf-8")


## 4. Run the Pipeline

This cell prepares the data, constructs meal events, and calculates the summary tables.


In [4]:
ensure_dirs()
dr1, meals = prepare_data()
participants, summary = summarize(meals)

print(f"Day-1 food rows: {len(dr1):,}")
print(f"Meal rows: {len(meals):,}")
print(f"Unique respondents in meal file: {meals['SEQN'].nunique():,}")
print(f"Output directory: {OUT_DIR}")


Day-1 food rows: 88,006
Meal rows: 28,890
Unique respondents in meal file: 6,679
Output directory: C:\Users\NathanSchweizer\Downloads\grocery_pricing\analysis\nhanes


## 5. Sample Sizes by SES Stratum

These are the strata used throughout the summary note.


In [5]:
participants


   stratum  participants  mean_age
       all          6679      42.3
 pir_3plus          2691      45.9
 pir_5plus          1436      46.1
pir_lt_1_3          1432      35.4


## 6. Inspect Top Archetypes

This quick view prints the leading meal archetypes for selected groups and occasions.


In [6]:
def show_top(summary, stratum, occasion, top_n=5):
    subset = summary[(summary["stratum"] == stratum) & (summary["occasion"] == occasion)].head(top_n)
    print(f"{stratum_label(stratum)} - {occasion.capitalize()}")
    for idx, row in enumerate(subset.itertuples(index=False), start=1):
        print(
            f"{idx}. {row.archetype} | share={row.weighted_share:.1%} | "
            f"participants={row.participants:,} | groups={compact_list(row.top_groups)}"
        )
    print()


show_top(summary, "all", "breakfast")
show_top(summary, "all", "lunch")
show_top(summary, "pir_5plus", "breakfast")
show_top(summary, "pir_5plus", "dinner")


All Respondents - Breakfast
1. other breakfast | share=32.3% | participants=1,887 | groups=sweet_baked, water, fruit, juice_smoothie
2. cereal or oatmeal breakfast | share=23.3% | participants=1,567 | groups=milk_dairy_drink, cereal_oatmeal, coffee_tea
3. coffee-first breakfast | share=16.5% | participants=1,025 | groups=coffee_tea, milk_dairy_drink, fruit
4. breakfast sandwich or burrito | share=9.5% | participants=559 | groups=sandwich, coffee_tea, water
5. coffee and pastry breakfast | share=9.3% | participants=617 | groups=coffee_tea, sweet_baked, bread

All Respondents - Lunch
1. other lunch | share=29.5% | participants=1,526 | groups=water, meat_poultry, fruit, cheese
2. sandwich lunch | share=28.5% | participants=1,514 | groups=sandwich, water, sweet_drink
3. salad lunch | share=11.5% | participants=591 | groups=salad, meat_poultry, vegetables
4. plate lunch | share=9.6% | participants=526 | groups=meat_poultry, grain_starch, sweet_drink
5. tortilla lunch | share=6.1% | particip

## 7. Regenerate the Markdown Note

This writes the formatted note to disk and previews the start of the file.


In [7]:
write_markdown(participants, summary)
preview_path = OUT_DIR / "nhanes_meal_archetypes.md"
print(preview_path)
print()
print("\n".join(preview_path.read_text(encoding="utf-8").splitlines()[:40]))


C:\Users\NathanSchweizer\Downloads\grocery_pricing\analysis\nhanes\nhanes_meal_archetypes.md

# NHANES Meal Archetypes (August 2021-August 2023)

This note summarizes a first-pass meal-archetype extraction using NHANES/WWEIA Day 1 dietary recall data and NHANES demographics.

## What This Is

- Meal unit: one respondent, one day, one eating occasion
- Source files: NHANES demographics plus WWEIA individual foods
- Weighting: Day 1 dietary weight (`WTDRD1`)
- SES split: poverty-income ratio (`INDFMPIR`)
- Caveat: these are heuristic meal archetypes, so the `other` buckets are still broad

## Sample Sizes

| Group | Participants | Mean age |
|---|---:|---:|
| All Respondents | 6,679 | 42.3 |
| Higher SES (PIR >= 3) | 2,691 | 45.9 |
| Highest SES (PIR >= 5) | 1,436 | 46.1 |
| Lower SES (PIR < 1.3) | 1,432 | 35.4 |

## All Respondents

- Participants: 6,679
- Mean age: 42.3

### Breakfast

1. **other breakfast** - 32.3% weighted share; 1,887 participants
   - Typical groups: sweet_baked, w